In [3]:
import os
import dotenv
import pymysql
import pandas as pd
import numpy as np

In [4]:
# la funzione load_dotenv legge il file .env e ne mette il contenuto
# nelle variabili d'ambiente del sistema operativo
dotenv.load_dotenv(dotenv_path=".env", override=True)

# tramite la libreria os ("operating system") leggiamo le variabili
# d'ambiente, e trasferiamo il contenuto in delle variabili Python
username = os.getenv("username")
password = os.getenv("password")
hostname = os.getenv("hostname")
database = os.getenv("database")

# creiamo la variabile "connection" che gestisce la connessione a database
connection = pymysql.connect(
    host=hostname,
    user=username,
    password=password,
    database=database,
    cursorclass=pymysql.cursors.DictCursor
)

In [5]:
cursor = connection.cursor()
query = "SELECT * FROM dimproduct;"
cursor.execute(query)
result = cursor.fetchall()
dimproduct = pd.DataFrame(result)

# arrotondiamo dealerprice + clip
dimproduct["DealerPrice"] = pd.to_numeric(dimproduct["DealerPrice"])
dimproduct["DealerPrice"].round(2)
dimproduct["DealerPrice"].round(0)

dimproduct["DealerPrice"].clip(lower=0, upper=1000)


0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
        ...   
601     60.744
602     72.894
603    323.994
604    323.994
605    323.994
Name: DealerPrice, Length: 606, dtype: float64

In [31]:
years = 5

guadagni = pd.DataFrame({
    "Mese": list("GFMAMGLASOND"*years),
    "Anno": np.repeat(list(range(years)), 12),
    "Valore": np.random.randint(800, 5000, 12*years)
})

guadagni["Valore"].cumsum()
guadagni.groupby("Anno")["Valore"].sum() # restituisce la somma di ogni anno

# Se invece volessero la "somma cumulativa degli anni" (anno 0, poi anno 0+1, poi anno 0+1+2...) si farebbe:
guadagni.groupby("Anno")["Valore"].sum().cumsum()


Anno
0     28807
1     57806
2     95229
3    133936
4    169404
Name: Valore, dtype: int64

In [13]:
cursor = connection.cursor()
queryc = "SELECT * FROM dimcustomer;"
cursor.execute(queryc)
result = cursor.fetchall()
dimcustomer = pd.DataFrame(result)

# nome e cognome
dimcustomer["FirstName"] = dimcustomer["FirstName"].str.lower()
dimcustomer["LastName"] = dimcustomer["LastName"].str.upper()

# split su mail 
dimcustomer["EmailSplit"] = dimcustomer["EmailAddress"].str.split(pat="@")
dimcustomer["UserMail"] = dimcustomer["EmailSplit"].str[0]
dimcustomer["DominioMail"] = dimcustomer["EmailSplit"].str[1]

# phone
dimcustomer["PhoneSplit"] = dimcustomer["Phone"].str.split()

# mail che contengono "21"
dimcustomer["Check21"] = dimcustomer["EmailAddress"].str.contains("21", regex=False)

# mail 20 o 10
dimcustomer["Check20"] = dimcustomer["EmailAddress"].str.contains("20", regex=False)
dimcustomer["Check10"] = dimcustomer["EmailAddress"].str.contains("10", regex=False)
mail_20_10 = dimcustomer[dimcustomer["Check20"] | dimcustomer["Check10"]]
mail_20_10["EmailAddress"]

# lunghezza e-mail
dimcustomer.EmailAddress.str.len().sort_values()
dimcustomer["LenEmail"] = dimcustomer["EmailAddress"].str.len()
lenmail = dimcustomer.sort_values("LenEmail")
lenmail[["EmailAddress", "LenEmail"]].head(5)
lenmail[["EmailAddress", "LenEmail"]].tail(5)

# modificare il dominio
dimcustomer.EmailAddress = dimcustomer.EmailAddress.str.replace("adventure-works.com", "aw-db.com")

# estrarre indirizzi street
filtro_street = dimcustomer["AddressLine1"].str.contains("Street")
indirizzi_street = dimcustomer[filtro_street]
indirizzi_street["AddressLine1"]


<class 'pandas.core.series.Series'>
Index: 512 entries, 7 to 18351
Series name: AddressLine1
Non-Null Count  Dtype 
--------------  ----- 
512 non-null    object
dtypes: object(1)
memory usage: 8.0+ KB


In [49]:
path = "C:/Users/giovanni.esposito/Desktop/EPICODE/M3 PYTHON/datasets/beginner_datasets/facebook.csv"
facebook = pd.read_csv(path)

# convertiamo in datetime
facebook.status_published = pd.to_datetime(facebook.status_published)

# otteniamo info specifiche sulle date transazioni
facebook["year"] = facebook.status_published.dt.year
facebook["month"] = facebook.status_published.dt.month
facebook["day"] = facebook.status_published.dt.day
facebook["day_week"] = facebook.status_published.dt.dayofweek
facebook["day_year"] = facebook.status_published.dt.dayofyear

# estraiamo i post
filtro2012 = facebook["year"] == 2012
facebook[filtro2012]

filtromay18 = (facebook["year"] == 2018) & (facebook["month"] == 5) 
facebook[filtromay18]

# confrontiamo i post del weekend vs intrasettimanali
filtrowe = facebook["day_week"] >= 5
filtroset = facebook["day_week"] < 5
n_post_we = facebook[filtrowe]["status_type"].count()
n_post_set = facebook[filtroset]["status_type"].count()

print("Il numero di post nel weekend è pari a ", n_post_we, "mentre il numero di post infrasettimanali è pari a ", n_post_set)

# primo e ultimo post ogni anno
for anno in facebook.year.unique():

    dati_anno = facebook[facebook["year"] == anno]

    print("Il primo post del", anno, "è", dati_anno["status_id"].head(1).iloc[0])

    print("L'ultimo post del", anno, "è",dati_anno["status_id"].tail(1).iloc[0])

# quanti post e quanti per ogni tipo
facebook.groupby("status_type")["status_type"].count() # 4 tipi di post, quanti per ogni tipo lo da l'output



Il numero di post nel weekend è pari a  2041 mentre il numero di post infrasettimanali è pari a  5009
Il primo post del 2018 è 246675545449582_1649696485147474
L'ultimo post del 2018 è 1050855161656896_1495116993897375
Il primo post del 2017 è 246675545449582_1530240427093081
L'ultimo post del 2017 è 1050855161656896_1158522910890120
Il primo post del 2016 è 246675545449582_1157109567739504
L'ultimo post del 2016 è 1050855161656896_1050858841656528
Il primo post del 2015 è 246675545449582_867129116737552
L'ultimo post del 2015 è 246675545449582_682315955218870
Il primo post del 2014 è 246675545449582_668776289906170
L'ultimo post del 2014 è 246675545449582_486434424807025
Il primo post del 2013 è 246675545449582_484017401715394
L'ultimo post del 2013 è 246675545449582_307286656055137
Il primo post del 2012 è 246675545449582_303160673134402
L'ultimo post del 2012 è 246675545449582_246677465449390


status_type
link        63
photo     4288
status     365
video     2334
Name: status_type, dtype: int64

In [61]:
pathp = "C:/Users/giovanni.esposito/Desktop/EPICODE/M3 PYTHON/datasets/beginner_datasets/pokemon.csv"
pokemon = pd.read_csv(pathp)

# controlliamo i valori nulli
pokemon.isnull().sum() # 386 valori nulli in colonna Type2 

# ha senso riempirli ? dipende dal senso di "Type 2": se ogni pokemon ha obbligatoriamente un type 2 allora si tratta di dati nulli anomali/problematici; se invece alcune categorie di pokemon non hanno il type 2 si tratta di null fisiologici/strutturali

# eliminiamo righe con dati nulli
pokemon_clean = pokemon.dropna()

In [90]:
patha = "C:/Users/giovanni.esposito/Desktop/EPICODE/M3 PYTHON/datasets/beginner_datasets/automobile.csv"
auto = pd.read_csv(patha)

# valori null
auto_null = auto.isnull().sum() 
filtro1 = auto_null != 0
print ("Di seguito un recap dei valori null: \n", auto_null[filtro1])

filtro2 = auto["num-of-doors"].isna()
auto[filtro2]

auto["num-of-doors"] = auto["num-of-doors"].fillna("not_available") 


Di seguito un recap dei valori null: 
 normalized-losses    37
num-of-doors          2
dtype: int64


In [93]:
import numpy as np, pandas as pd

temp = pd.DataFrame({
    "Giorno": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "Temperature": [18, 19, 18, np.nan, 21, 20, 20, np.nan, 21, 23, np.nan, 23, 24]
})

# l'interpolazione mi sembra tendenzialmente la scelta migliore, soprattutto in un caso di dati non randomici ma temporali e conseguenziali
temp_fill = temp.interpolate() 
temp_fill

,Giorno,Temperature
0,0,18.0
1,1,19.0
2,2,18.0
3,3,19.5
4,4,21.0
5,5,20.0
6,6,20.0
7,7,20.5
8,8,21.0
9,9,23.0


In [112]:
listadataset = os.listdir("C:/Users/giovanni.esposito/Desktop/EPICODE/M3 PYTHON/datasets/beginner_datasets/")
pathl = "C:/Users/giovanni.esposito/Desktop/EPICODE/M3 PYTHON/datasets/beginner_datasets/"

saltati = []

for dataset in listadataset:
    try:  # il try / exepct l'ho fatto scrivere AI perchè c'è il problema di csv con encoding non utf-8
        dataset_csv = pd.read_csv(pathl + dataset)
        totale_null = dataset_csv.isnull().sum().sum()
        if totale_null > 0:    # questo evita una spataffiata di tabelle con indicazione di tutte le colonne
            print(dataset, "-> null totali:", totale_null)
    except:
        saltati.append(dataset)

print("---")
print("File saltati:", saltati)   #il maledetto che non è utf-8

automobile.csv -> null totali: 39
elections.csv -> null totali: 52
france.csv -> null totali: 66
hepatitis.csv -> null totali: 153
house.csv -> null totali: 7829
income.csv -> null totali: 4262
mice.csv -> null totali: 1396
nba.csv -> null totali: 11
pokemon.csv -> null totali: 386
population.csv -> null totali: 12
seeds.csv -> null totali: 4
traffic.csv -> null totali: 48143
wikipedia.csv -> null totali: 68
---
File saltati: ['pycaret_datasets.xlsx']
